In [13]:
# Core libraries
import os
import glob
import re

# Data handling
import pandas as pd
import numpy as np

# OCR
import easyocr

# cv2
import cv2

In [14]:
image_folder = r"\\nc_sensitive1\mortgage\Shared\Shared\Departments\Secondary Marketing West Coast\_Daily MBS"

if not os.path.exists(image_folder):
    raise FileNotFoundError(f"Image folder not found: {image_folder}")

In [15]:
'''image_files = sorted(glob.glob(os.path.join(image_folder, "*.jpg")))

print(f"Number of JPG files found: {len(image_files)}")

for file in image_files:
    print(os.path.basename(file))'''

image_files = sorted(glob.glob(os.path.join(image_folder, "*.jpg")))
# exclude preprocessed files/folder just in case
image_files = [
   f for f in image_files
   if "_preprocessed" not in f and not os.path.basename(f).endswith("_bw.jpg")
]
print(f"Original JPG files found: {len(image_files)}")
for f in image_files:
   print(os.path.basename(f))

Original JPG files found: 4
UMBS_15YR_3.0-4.5.jpg
UMBS_15YR_5.0-6.5.jpg
UMBS_GNMAII_30YR_3.0-5.5.jpg
UMBS_GNMAII_30YR_4.5-7.0.jpg


In [16]:
preprocessed_folder = os.path.join(image_folder, "_preprocessed")
os.makedirs(preprocessed_folder, exist_ok=True)
def preprocess_mbs_image(img_path):
   img = cv2.imread(img_path)
   # upscale 2x
   img = cv2.resize(img, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)
   # grayscale
   gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
   # increase contrast / binary threshold
   _, bw = cv2.threshold(gray, 120, 255, cv2.THRESH_BINARY)
   out_path = os.path.join(
       preprocessed_folder,
       os.path.basename(img_path).replace(".jpg", "_bw.jpg")
   )
   cv2.imwrite(out_path, bw)
   return out_path

In [17]:
try:
    reader = easyocr.Reader(["en"], gpu=True)
    print("EasyOCR reader loaded with GPU enabled.")
except Exception as e:
    print(f"GPU failed, falling back to CPU. Error: {e}")
    reader = easyocr.Reader(["en"], gpu=False)
    print("EasyOCR reader loaded with CPU.")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.


EasyOCR reader loaded with GPU enabled.


In [18]:
full_price_pattern = re.compile(r"^\d{2,3}-\d{2}\+?\s*/\s*\d{1,2}\+?$")
left_price_pattern = re.compile(r"^\d{2,3}-\d{2}\+?$")
right_tick_pattern = re.compile(r"^/?\d{1,2}\+?$")

month_names = "Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec"

row_label_pattern = re.compile(
    rf"^(({month_names})|({month_names})/({month_names}))$",
    re.IGNORECASE
)

header_pattern = re.compile(
    r"^(\d\.\d|UMBS|GNMAII)$",
    re.IGNORECASE
)

In [19]:
records = []

for img_path in image_files:
    print(f"\n===== Processing Image: {img_path} =====")

    ocr_path = preprocess_mbs_image(img_path)
    print("OCR Input: ", ocr_path)

    result = reader.readtext(
        ocr_path,
        allowlist="0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz-/+.",
        paragraph=False
        )

    for bbox, text, conf in result:
        raw_text = str(text).strip()

        if raw_text == "":
            continue

        (x1, y1), (x2, y2), (x3, y3), (x4, y4) = bbox
        xs = [x1, x2, x3, x4]
        ys = [y1, y2, y3, y4]

        records.append({
            "image": img_path,
            "image_name": os.path.basename(img_path),
            "raw_text": raw_text,
            "conf": conf,
            "x_center": np.mean(xs),
            "y_center": np.mean(ys),
            "x_min": min(xs),
            "x_max": max(xs),
            "y_min": min(ys),
            "y_max": max(ys),
            "bbox": bbox
        })

df_raw = pd.DataFrame(records)

print(f"Raw OCR boxes captured: {len(df_raw)}")
display(df_raw.head(50))


===== Processing Image: \\nc_sensitive1\mortgage\Shared\Shared\Departments\Secondary Marketing West Coast\_Daily MBS\UMBS_15YR_3.0-4.5.jpg =====
OCR Input:  \\nc_sensitive1\mortgage\Shared\Shared\Departments\Secondary Marketing West Coast\_Daily MBS\_preprocessed\UMBS_15YR_3.0-4.5_bw.jpg


C:\Users\iqw67\AppData\Roaming\Python\Python312\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



===== Processing Image: \\nc_sensitive1\mortgage\Shared\Shared\Departments\Secondary Marketing West Coast\_Daily MBS\UMBS_15YR_5.0-6.5.jpg =====
OCR Input:  \\nc_sensitive1\mortgage\Shared\Shared\Departments\Secondary Marketing West Coast\_Daily MBS\_preprocessed\UMBS_15YR_5.0-6.5_bw.jpg

===== Processing Image: \\nc_sensitive1\mortgage\Shared\Shared\Departments\Secondary Marketing West Coast\_Daily MBS\UMBS_GNMAII_30YR_3.0-5.5.jpg =====
OCR Input:  \\nc_sensitive1\mortgage\Shared\Shared\Departments\Secondary Marketing West Coast\_Daily MBS\_preprocessed\UMBS_GNMAII_30YR_3.0-5.5_bw.jpg

===== Processing Image: \\nc_sensitive1\mortgage\Shared\Shared\Departments\Secondary Marketing West Coast\_Daily MBS\UMBS_GNMAII_30YR_4.5-7.0.jpg =====
OCR Input:  \\nc_sensitive1\mortgage\Shared\Shared\Departments\Secondary Marketing West Coast\_Daily MBS\_preprocessed\UMBS_GNMAII_30YR_4.5-7.0_bw.jpg
Raw OCR boxes captured: 303


,image,image_name,raw_text,conf,x_center,y_center,x_min,x_max,y_min,y_max,bbox
0,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,3.OUMBS,0.504799,380.00,70.0,278.000000,482.000000,44.000000,96.000000,"[[278, 44], [482, 44], [482, 96], [278, 96]]"
1,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,4.0UMBS,0.860949,800.00,68.0,698.000000,902.000000,42.000000,94.000000,"[[698, 42], [902, 42], [902, 94], [698, 94]]"
2,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,Jun,0.999506,74.00,135.0,29.000000,119.000000,115.000000,155.000000,"[[29, 115], [119, 115], [119, 155], [29, 155]]"
3,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,94-07,0.999986,351.00,134.0,278.000000,424.000000,110.000000,158.000000,"[[278, 110], [424, 110], [424, 158], [278, 158]]"
4,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,115,0.955280,493.00,133.0,449.000000,537.000000,111.000000,155.000000,"[[449, 111], [537, 111], [537, 155], [449, 155]]"
5,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,96-20+/23,0.848338,855.50,133.0,725.000000,986.000000,107.000000,159.000000,"[[725, 107], [986, 107], [986, 159], [725, 159]]"
6,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,Jul,0.999798,72.00,196.0,29.000000,115.000000,175.000000,217.000000,"[[29, 175], [115, 175], [115, 217], [29, 217]]"
7,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,94-06,0.999976,353.00,196.0,278.000000,428.000000,172.000000,220.000000,"[[278, 172], [428, 172], [428, 220], [278, 220]]"
8,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,114,0.977725,492.00,197.0,448.000000,536.000000,172.000000,222.000000,"[[448, 172], [536, 172], [536, 222], [448, 222]]"
9,\\nc_sensitive1\mortgage\Shared\Shared\Departm...,UMBS_15YR_3.0-4.5.jpg,96-18+/20,0.999954,855.50,195.5,725.000000,986.000000,169.000000,222.000000,"[[725, 169], [986, 169], [986, 222], [725, 222]]"


In [20]:
def normalize_ocr_text(text):
    if not isinstance(text, str):
        return None

    s = text.strip()
    s = s.replace(" ", "")
    s = s.replace("T", "/").replace("t", "/")
    s = s.replace("\\", "/")

    # Fix impossible OCR pattern like 99-28+-/29+ -> 99-28+/29+
    s = re.sub(r"\+-+(?=/)", "+", s)
    
    # Only fix l/I/| -> 1 for price/tick-looking strings
    looks_like_price_or_tick = (
        any(ch.isdigit() for ch in s)
        and ("-" in s or "/" in s or "+" in s))
        
    if looks_like_price_or_tick:
        s = s.replace("l", "1").replace("I", "1").replace("|", "1")
        s = re.sub(r"\s+", "", s)


    # Defensive OCR correction:
    # EasyOCR may read slash ticks as 7/71, e.g. /20 -> 720 or 7120.
    # Only convert when the remaining value is a valid MBS tick from 00 to 32.
    slash_misread = re.fullmatch(r"7(?:1)?(\d{2})(\+?)", s)
    if slash_misread:
        tick = slash_misread.group(1)
        plus = slash_misread.group(2)
        if 0 <= int(tick) <= 32:
            s = f"/{tick}{plus}"

    # Same slash correction when attached to a price token:
    # 101-147120 -> 101-14/20
    attached_slash_misread = re.fullmatch(
        r"(\d{2,3}-\d{2}\+?)7(?:1)?(\d{2})(\+?)",
        s
    )
    if attached_slash_misread:
        left_price = attached_slash_misread.group(1)
        tick = attached_slash_misread.group(2)
        plus = attached_slash_misread.group(3)
        if 0 <= int(tick) <= 32:
            s = f"{left_price}/{tick}{plus}"

    # Existing protection: 716 -> /16, 107+ -> /07+
    if re.fullmatch(r"[17]\d{2}\+?", s):
        plus = "+" if s.endswith("+") else ""
        digits = s.replace("+", "")
        tick = digits[1:]
        if tick.isdigit() and 0 <= int(tick) <= 32:
            s = f"/{tick}{plus}"

    return s if s else None


def classify_ocr_text(text):
    if not isinstance(text, str):
        return "other"
    s = text.strip()
    if full_price_pattern.match(s):
        return "full_price"
    if left_price_pattern.match(s):
        return "left_price"
    if right_tick_pattern.match(s):
        return "right_tick"
    if row_label_pattern.match(s):
        return "row_label"
    if header_pattern.match(s):
        return "header"
    return "other"


def split_full_price(text):
    if not isinstance(text, str):
        return None, None

    s = text.strip()

    if "/" not in s:
        return s, None

    price, tick = s.split("/", 1)
    return price, "/" + tick
    
df_ocr = df_raw.copy()
df_ocr["text"] = df_ocr["raw_text"].apply(normalize_ocr_text)
df_ocr["text_type"] = df_ocr["text"].apply(classify_ocr_text)

def assign_row_ids(group, row_tol=30):
   group = group.sort_values("y_center").copy()
   rows = []
   row_centers = []
   for _, r in group.iterrows():
       y = r["y_center"]
       if len(row_centers) == 0:
           row_centers.append(y)
           rows.append(1)
           continue
       distances = [abs(y - yc) for yc in row_centers]
       min_idx = int(np.argmin(distances))
       if distances[min_idx] <= row_tol:
           rows.append(min_idx + 1)
           row_centers[min_idx] = np.mean([row_centers[min_idx], y])
       else:
           row_centers.append(y)
           rows.append(len(row_centers))
   group["row"] = rows
   return group

df_ocr = (
   df_ocr
   .groupby("image", group_keys=False)
   .apply(assign_row_ids)
)
df_ocr = df_ocr.sort_values(["image", "row", "x_center"]).reset_index(drop=True)
display(df_ocr[["image_name", "row", "raw_text", "text", "text_type", "x_center", "y_center"]])


C:\Users\iqw67\AppData\Local\Temp\ipykernel_34820\994672659.py:114: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_row_ids)


,image_name,row,raw_text,text,text_type,x_center,y_center
0,UMBS_15YR_3.0-4.5.jpg,1,3.OUMBS,3.OUMBS,other,380.0,70.0
1,UMBS_15YR_3.0-4.5.jpg,1,4.0UMBS,4.0UMBS,other,800.0,68.0
2,UMBS_15YR_3.0-4.5.jpg,2,Jun,Jun,row_label,74.0,135.0
3,UMBS_15YR_3.0-4.5.jpg,2,94-07,94-07,left_price,351.0,134.0
4,UMBS_15YR_3.0-4.5.jpg,2,115,/15,right_tick,493.0,133.0
...,...,...,...,...,...,...,...
298,UMBS_GNMAII_30YR_4.5-7.0.jpg,12,103-06+/18,103-06+/18,full_price,2066.0,707.0
299,UMBS_GNMAII_30YR_4.5-7.0.jpg,13,Sep,Sep,row_label,63.5,770.5
300,UMBS_GNMAII_30YR_4.5-7.0.jpg,13,97-08,97-08,left_price,345.0,768.0
301,UMBS_GNMAII_30YR_4.5-7.0.jpg,13,111,/11,right_tick,483.0,769.0


In [21]:
'''quote_rows = []
for img_path, img_df in df_ocr.groupby("image"):
   img_df = img_df.sort_values(["row", "x_center"]).copy()
   for row_id, row_df in img_df.groupby("row"):
       row_df = row_df.sort_values("x_center").copy()
       items = row_df[
           row_df["text_type"].isin(["row_label", "full_price", "left_price", "right_tick"])
       ].copy()
       month = None
       pending_price = None
       for _, item in items.iterrows():
           text = item["text"]
           text_type = item["text_type"]
           if text_type == "row_label":
               month = text
               continue
           if text_type == "full_price":
               price, tick = split_full_price(text)
               quote_rows.append({
                   "image_name": os.path.basename(img_path),
                   "row": row_id,
                   "month": month,
                   "price": price,
                   "ticks": tick,
                   "x_center": item["x_center"],
                   "y_center": item["y_center"]
               })
               continue
           if text_type == "left_price":
               pending_price = text
               continue
           if text_type == "right_tick" and pending_price is not None:
               quote_rows.append({
                   "image_name": os.path.basename(img_path),
                   "row": row_id,
                   "month": month,
                   "price": pending_price,
                   "ticks": text,
                   "x_center": item["x_center"],
                   "y_center": item["y_center"]
               })
               pending_price = None

df_final = pd.DataFrame(quote_rows)
df_final = df_final.sort_values(
   ["image_name", "row", "x_center"]
).reset_index(drop=True)
display(df_final)'''

'quote_rows = []\nfor img_path, img_df in df_ocr.groupby("image"):\n   img_df = img_df.sort_values(["row", "x_center"]).copy()\n   for row_id, row_df in img_df.groupby("row"):\n       row_df = row_df.sort_values("x_center").copy()\n       items = row_df[\n           row_df["text_type"].isin(["row_label", "full_price", "left_price", "right_tick"])\n       ].copy()\n       month = None\n       pending_price = None\n       for _, item in items.iterrows():\n           text = item["text"]\n           text_type = item["text_type"]\n           if text_type == "row_label":\n               month = text\n               continue\n           if text_type == "full_price":\n               price, tick = split_full_price(text)\n               quote_rows.append({\n                   "image_name": os.path.basename(img_path),\n                   "row": row_id,\n                   "month": month,\n                   "price": price,\n                   "ticks": tick,\n                   "x_center": item["x

In [22]:
quote_rows = []
def append_standalone_left_price(pending_item, img_path, row_id, month):
   """
   Handles cases like 104-01+ where OCR gives us only a left_price
   and no separate right_tick appears after it.
   """
   text = pending_item["text"]
   price, tick = split_full_price(text)
   quote_rows.append({
       "image_name": os.path.basename(img_path),
       "row": row_id,
       "month": month,
       "price": price,
       "ticks": tick,
       "x_center": pending_item["x_center"],
       "y_center": pending_item["y_center"]
   })

for img_path, img_df in df_ocr.groupby("image"):
   img_df = img_df.sort_values(["row", "x_center"]).copy()
   for row_id, row_df in img_df.groupby("row"):
       row_df = row_df.sort_values("x_center").copy()
       items = row_df[
           row_df["text_type"].isin(["row_label", "full_price", "left_price", "right_tick"])
       ].copy()
       month = None
       pending_item = None
       for _, item in items.iterrows():
           text = item["text"]
           text_type = item["text_type"]
           if text_type == "row_label":
               month = text
               continue
           if text_type == "full_price":
               # If we had a pending left_price before this,
               # it means no right_tick came for it.
               if pending_item is not None:
                   append_standalone_left_price(pending_item, img_path, row_id, month)
                   pending_item = None
               price, tick = split_full_price(text)
               quote_rows.append({
                   "image_name": os.path.basename(img_path),
                   "row": row_id,
                   "month": month,
                   "price": price,
                   "ticks": tick,
                   "x_center": item["x_center"],
                   "y_center": item["y_center"]
               })
               continue
           if text_type == "left_price":
               # If another left_price appears before a right_tick,
               # flush the old one as standalone.
               if pending_item is not None:
                   append_standalone_left_price(pending_item, img_path, row_id, month)
               pending_item = item
               continue
           if text_type == "right_tick":
               if pending_item is not None:
                   quote_rows.append({
                       "image_name": os.path.basename(img_path),
                       "row": row_id,
                       "month": month,
                       "price": pending_item["text"],
                       "ticks": text,
                       "x_center": pending_item["x_center"],
                       "y_center": pending_item["y_center"]
                   })
                   pending_item = None
               continue
       # End of row:
       # If a left_price was left hanging, then it must be standalone like 104-01+
       if pending_item is not None:
           append_standalone_left_price(pending_item, img_path, row_id, month)
           pending_item = None

df_final = pd.DataFrame(quote_rows)
df_final = df_final.sort_values(
   ["image_name", "row", "x_center"]
).reset_index(drop=True)
display(df_final)

,image_name,row,month,price,ticks,x_center,y_center
0,UMBS_15YR_3.0-4.5.jpg,2,Jun,94-07,/15,351.0,134.0
1,UMBS_15YR_3.0-4.5.jpg,2,Jun,96-20+,/23,855.5,133.0
2,UMBS_15YR_3.0-4.5.jpg,3,Jul,94-06,/14,353.0,196.0
3,UMBS_15YR_3.0-4.5.jpg,3,Jul,96-18+,/20,855.5,195.5
4,UMBS_15YR_3.0-4.5.jpg,4,Aug,94-06,/14,352.5,257.0
...,...,...,...,...,...,...,...
87,UMBS_GNMAII_30YR_4.5-7.0.jpg,12,Aug,101-14,/19+,1338.0,706.0
88,UMBS_GNMAII_30YR_4.5-7.0.jpg,12,Aug,105-16+,/19,1730.5,705.5
89,UMBS_GNMAII_30YR_4.5-7.0.jpg,12,Aug,103-06+,/18,2066.0,707.0
90,UMBS_GNMAII_30YR_4.5-7.0.jpg,13,Sep,97-08,/11,345.0,768.0


In [23]:

def parse_tick_value(tick):
   if not isinstance(tick, str):
       return np.nan
   s = tick.strip().replace("/", "")
   has_plus = s.endswith("+")
   s = s.replace("+", "")
   if not s.isdigit():
       return np.nan
   tick_num = int(s)
   if tick_num < 0 or tick_num > 32:
       return np.nan
   return tick_num + 0.5 if has_plus else tick_num


def parse_price_base(price):
   if not isinstance(price, str):
       return pd.Series([np.nan, np.nan])
   s = price.strip()
   m = re.fullmatch(r"(\d{2,3})-(\d{2})(\+?)", s)
   if m:
       base = int(m.group(1))
       price_tick = int(m.group(2))
       if m.group(3) == "+":
           price_tick += 0.5
       return pd.Series([base, price_tick])
   m = re.fullmatch(r"-(\d{2,3})(\+?)", s)
   if m:
       price_tick = -int(m.group(1))
       if m.group(2) == "+":
           price_tick -= 0.5
       return pd.Series([0, price_tick])
   m = re.fullmatch(r"(\d{1,3})(\+?)", s)
   if m:
       price_tick = int(m.group(1))
       if m.group(2) == "+":
           price_tick += 0.5
       return pd.Series([0, price_tick])
   return pd.Series([np.nan, np.nan])


def calculate_ask_decimal(base, price_tick, ask_tick):
   """
   Convert MBS price/tick fields into decimal price.

   Example:
       price = 101-14, ticks = /20 -> 101 + 20/32 = 101.625
   """
   if pd.isna(base) or pd.isna(price_tick) or pd.isna(ask_tick):
       return np.nan

   ask_base = base

   # Rollover case, e.g. 99-30 with /03+ means 100 + 3.5/32.
   if ask_tick < price_tick:
       ask_base += 1

   return ask_base + (ask_tick / 32)


def add_months(month_start, n):
    y = month_start.year + (month_start.month - 1 + n) // 12
    m = (month_start.month - 1 + n) % 12 + 1
    return pd.Timestamp(year=y, month=m, day=1)


def expected_month_labels(month_count=4):
    """
    Current dynamic month plus the next months, capped to exactly month_count labels.
    Example on a Jun run: Jun, Jul, Aug, Sep.
    This keeps transition-month rows alive without creating an extra Oct column/month.
    """
    start = pd.Timestamp.today().normalize().replace(day=1)
    return [add_months(start, i).strftime("%b") for i in range(month_count)]


def get_expected_layout(image_name, section):
    """
    Full expected left-to-right quote structure for each image/section.

    Important:
    - Product/rate must not be assigned by simple row cumcount.
    - If a left column is missing, row order shifts and GNMAII/UMBS get mislabeled.
    - We assign actual OCR quotes to the nearest x-anchor, then rebuild a complete
      month x product x coupon grid so missing values remain blank rows.
    """
    name = str(image_name)

    if "3.0-5.5" in name:
        rates = [3.5, 4.5, 5.5] if section == 1 else [4.0, 5.0, 6.0]
        products = ["UMBS", "GNMAII"]
    elif "4.5-7.0" in name:
        rates = [4.5, 5.5, 6.5] if section == 1 else [5.0, 6.0, 7.0]
        products = ["UMBS", "GNMAII"]
    elif "3.0-4.5" in name:
        rates = [3.0, 4.0] if section == 1 else [3.5, 4.5]
        products = ["UMBS"]
    elif "5.0-6.5" in name:
        rates = [5.0, 6.0] if section == 1 else [5.5, 6.5]
        products = ["UMBS"]
    else:
        return []

    layout = []
    pos = 1
    for rate in rates:
        for product in products:
            layout.append({
                "quote_position": pos,
                "rate": rate,
                "product": product
            })
            pos += 1
    return layout


def infer_x_anchors_for_group(g, expected_n):
    """
    Infer stable expected x-centers for the full image section.
    Uses all rows in the section, so one missing month/security row does not shift mapping.
    """
    xs = sorted(pd.to_numeric(g["x_center"], errors="coerce").dropna().tolist())
    if not xs or expected_n <= 0:
        return [np.nan] * expected_n

    # If there are fewer OCR detections than expected columns, keep existing anchors
    # and pad missing anchor columns with NaN. Placeholders will still be created.
    if len(xs) <= expected_n:
        anchors = xs + [np.nan] * (expected_n - len(xs))
        return anchors[:expected_n]

    anchors = np.quantile(xs, np.linspace(0, 1, expected_n))

    for _ in range(25):
        buckets = [[] for _ in range(expected_n)]
        for x in xs:
            idx = int(np.nanargmin([abs(x - a) for a in anchors]))
            buckets[idx].append(x)
        new_anchors = np.array([
            np.median(bucket) if bucket else anchors[i]
            for i, bucket in enumerate(buckets)
        ])
        if np.allclose(new_anchors, anchors, equal_nan=True):
            break
        anchors = new_anchors

    return sorted([float(a) if not pd.isna(a) else np.nan for a in anchors], key=lambda v: 999999 if pd.isna(v) else v)


# 1) Calculate decimal fields on actual OCR rows.
df_final[["price_base", "price_tick"]] = df_final["price"].apply(parse_price_base)
df_final["ask_tick_value"] = df_final["ticks"].apply(parse_tick_value)
df_final["decimal_price"] = df_final.apply(
   lambda r: calculate_ask_decimal(
       r["price_base"],
       r["price_tick"],
       r["ask_tick_value"]
   ),
   axis=1
)

# 2) Section: top block vs bottom block.
df_final["section"] = df_final["row"].apply(lambda r: 1 if r <= 5 else 2)

# 3) Build expected x anchors by image + section.
anchor_rows = []
for (image_name, section), g in df_final.groupby(["image_name", "section"]):
    layout = get_expected_layout(image_name, section)
    anchors = infer_x_anchors_for_group(g, len(layout))
    for idx, item in enumerate(layout, start=1):
        anchor_rows.append({
            "image_name": image_name,
            "section": section,
            "quote_position": idx,
            "expected_x": anchors[idx - 1] if idx - 1 < len(anchors) else np.nan,
            "rate": item["rate"],
            "product": item["product"]
        })

anchor_df = pd.DataFrame(anchor_rows)


def assign_nearest_quote_column(row):
    candidates = anchor_df[
        (anchor_df["image_name"] == row["image_name"]) &
        (anchor_df["section"] == row["section"]) &
        (anchor_df["expected_x"].notna())
    ].copy()
    if candidates.empty or pd.isna(row["x_center"]):
        return pd.Series([np.nan, np.nan, np.nan, np.nan])

    candidates["x_distance"] = (candidates["expected_x"] - row["x_center"]).abs()
    best = candidates.sort_values("x_distance").iloc[0]
    return pd.Series([
        best["quote_position"],
        best["rate"],
        best["product"],
        best["expected_x"]
    ])


# 4) Assign actual OCR quotes by x-position, not by row order.
df_final[["quote_position", "rate", "product", "expected_x"]] = df_final.apply(
    assign_nearest_quote_column,
    axis=1
)

# 5) Rebuild the COMPLETE expected grid, then left-merge actual OCR values into it.
#    This is the part that preserves blank placeholder rows for missing months/coupons/products.
#    Exactly 4 dynamic months only: current month + next 3 months.
months = expected_month_labels(month_count=4)
placeholder_rows = []
for (image_name, section), g in df_final.groupby(["image_name", "section"]):
    section_anchors = anchor_df[
        (anchor_df["image_name"] == image_name) &
        (anchor_df["section"] == section)
    ].copy()
    if section_anchors.empty:
        continue

    for month in months:
        for _, a in section_anchors.sort_values("quote_position").iterrows():
            placeholder_rows.append({
                "image_name": image_name,
                "section": section,
                "month": month,
                "quote_position": a["quote_position"],
                "rate": a["rate"],
                "product": a["product"],
                "expected_x": a["expected_x"],
                "is_placeholder": 1
            })

expected_grid = pd.DataFrame(placeholder_rows)

actual = df_final.copy()
actual["is_placeholder"] = 0

# If OCR creates duplicate assignments to the same expected slot, keep the closest to anchor.
actual["x_anchor_distance"] = (actual["x_center"] - actual["expected_x"]).abs()
actual = actual.sort_values(["image_name", "section", "month", "quote_position", "x_anchor_distance"])
actual = actual.drop_duplicates(["image_name", "section", "month", "quote_position"], keep="first")

merge_keys = ["image_name", "section", "month", "quote_position", "rate", "product", "expected_x"]
value_cols = [c for c in actual.columns if c not in merge_keys + ["is_placeholder"]]

df_final = expected_grid.merge(
    actual[merge_keys + value_cols],
    on=merge_keys,
    how="left"
)

# Anything that successfully matched actual OCR is not a placeholder.
df_final["is_placeholder"] = np.where(df_final["price"].notna() | df_final["ticks"].notna(), 0, 1)

# Reorder columns so the important audit fields are easy to see in Python New.
front_cols = [
    "image_name", "section", "month", "product", "rate", "quote_position",
    "price", "ticks", "decimal_price", "is_placeholder", "row", "x_center", "expected_x", "y_center"
]
front_cols = [c for c in front_cols if c in df_final.columns]
other_cols = [c for c in df_final.columns if c not in front_cols]
df_final = df_final[front_cols + other_cols]

# Sort months in true calendar/output order, not alphabetical OCR/order.
month_order_map = {m: i for i, m in enumerate(months)}
df_final["month_sort"] = df_final["month"].map(month_order_map)

df_final = df_final.sort_values(
    ["image_name", "section", "month_sort", "quote_position"]
).drop(columns=["month_sort"]).reset_index(drop=True)

# Audits.
decimal_audit = df_final[(df_final["is_placeholder"] == 0) & (df_final["decimal_price"].isna())].copy()
if not decimal_audit.empty:
   print("WARNING: decimal_price did not calculate for these non-placeholder rows:")
   display(decimal_audit[["image_name", "section", "month", "product", "rate", "price", "ticks", "price_base", "price_tick", "ask_tick_value"]])
else:
   print("Decimal price audit passed for all non-placeholder rows.")

print("Expected dynamic month rows:", months)
print("Placeholder rows created:", int(df_final["is_placeholder"].sum()))
print("Column-anchor audit sample: Jun rows")
display(
    df_final[df_final["month"].astype(str).str.lower() == "jun"]
    [["image_name", "section", "month", "product", "rate", "quote_position", "price", "ticks", "decimal_price", "is_placeholder", "x_center", "expected_x"]]
    .sort_values(["image_name", "section", "quote_position"])
)

display(df_final)


,image_name,row,month,price,ticks,price_base,price_tick,ask_tick_value
54,UMBS_GNMAII_30YR_3.0-5.5.jpg,13,Sep,92-10+,None,92.0,10.5,NaN


,image_name,row,month,price,ticks,x_center,y_center,price_base,price_tick,ask_tick_value,decimal_price,section,quote_position,rate
0,UMBS_15YR_3.0-4.5.jpg,2,Jun,94-07,/15,351.0,134.0,94.0,7.0,15.0,94.468750,1,1,3.0
1,UMBS_15YR_3.0-4.5.jpg,2,Jun,96-20+,/23,855.5,133.0,96.0,20.5,23.0,96.718750,1,2,4.0
2,UMBS_15YR_3.0-4.5.jpg,3,Jul,94-06,/14,353.0,196.0,94.0,6.0,14.0,94.437500,1,1,3.0
3,UMBS_15YR_3.0-4.5.jpg,3,Jul,96-18+,/20,855.5,195.5,96.0,18.5,20.0,96.625000,1,2,4.0
4,UMBS_15YR_3.0-4.5.jpg,4,Aug,94-06,/14,352.5,257.0,94.0,6.0,14.0,94.437500,1,1,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
87,UMBS_GNMAII_30YR_4.5-7.0.jpg,12,Aug,101-14,/19+,1338.0,706.0,101.0,14.0,19.5,101.609375,2,4,6.0
88,UMBS_GNMAII_30YR_4.5-7.0.jpg,12,Aug,105-16+,/19,1730.5,705.5,105.0,16.5,19.0,105.593750,2,5,7.0
89,UMBS_GNMAII_30YR_4.5-7.0.jpg,12,Aug,103-06+,/18,2066.0,707.0,103.0,6.5,18.0,103.562500,2,6,7.0
90,UMBS_GNMAII_30YR_4.5-7.0.jpg,13,Sep,97-08,/11,345.0,768.0,97.0,8.0,11.0,97.343750,2,1,5.0


In [24]:
with pd.ExcelWriter('//nc_sensitive1/mortgage/Shared/Shared/Departments/Secondary Marketing West Coast/_Daily MBS/Daily_MBS_MASTER_Final_Dynamic_v5.xlsx', mode = 'a', if_sheet_exists= "replace", engine = 'openpyxl') as writer:
    df_final.to_excel(writer, sheet_name = 'Python New', index = False)